In [1]:
import os
import xml.etree.ElementTree as ET
import rdflib
from dotenv import load_dotenv
from rdflib import Graph
from rdflib_neo4j import Neo4jStoreConfig, Neo4jStore

네임스페이스 자동 추출

In [3]:
print("go.owl 파일에서 네임스페이스를 동적으로 추출합니다...")
custom_prefixes = {}

# iterparse를 사용해 메모리에 전체 파일을 올리지 않고 네임스페이스 선언부만 빠르게 스캔
for event, elem in ET.iterparse("../data/go.owl", events=('start-ns',)):
    prefix, uri = elem
    if prefix:  # 기본(빈) 네임스페이스 제외
        custom_prefixes[prefix] = uri

go.owl 파일에서 네임스페이스를 동적으로 추출합니다...


In [4]:
len(custom_prefixes)

20

In [5]:
custom_prefixes

{'EC': 'http://purl.uniprot.org/enzyme/',
 'TC': 'http://identifiers.org/tcdb/',
 'VZ': 'https://viralzone.expasy.org/',
 'dc': 'http://purl.org/dc/elements/1.1/',
 'go': 'http://purl.obolibrary.org/obo/go#',
 'obo': 'http://purl.obolibrary.org/obo/',
 'owl': 'http://www.w3.org/2002/07/owl#',
 'rdf': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
 'xml': 'http://www.w3.org/XML/1998/namespace',
 'xsd': 'http://www.w3.org/2001/XMLSchema#',
 'RHEA': 'http://rdf.rhea-db.org/',
 'rdfs': 'http://www.w3.org/2000/01/rdf-schema#',
 'skos': 'http://www.w3.org/2004/02/skos/core#',
 'terms': 'http://purl.org/dc/terms/',
 'MetaCyc': 'http://identifiers.org/metacyc.reaction/',
 'Reactome': 'http://identifiers.org/reactome/',
 'oboInOwl': 'http://www.geneontology.org/formats/oboInOwl#',
 'KEGG_REACTION': 'http://www.kegg.jp/entry/',
 'UM-BBD_pathwayID': 'http://identifiers.org/umbbd.pathway/',
 'UM-BBD_reactionID': 'http://identifiers.org/umbbd.reaction/'}

In [ ]:
# 1. .env 파일에서 환경 변수 로드
load_dotenv()

neo4j_uri = os.getenv("NEO4J_URI", "bolt://localhost:7687")
neo4j_user = os.getenv("NEO4J_USER", "neo4j")
neo4j_password = os.getenv("NEO4J_PASSWORD")
neo4j_db = os.getenv("NEO4J_DATABASE", "neo4j")

if not neo4j_password:
    raise ValueError(".env 파일에 NEO4J_PASSWORD 환경 변수가 설정되지 않았습니다.")

auth_data = {
    "uri": neo4j_uri,
    "database": neo4j_db,
    "user": neo4j_user,
    "pwd": neo4j_password,
}

# 2. 💡 [핵심] 유저 아이디어 적용: XML 파일에서 네임스페이스 동적 추출해서 custom_prefixes에 앞에서 저장함
print(f"✅ 총 {len(custom_prefixes)}개의 네임스페이스 매핑 완료!")

# 3. 동적으로 생성된 네임스페이스 딕셔너리를 설정에 주입
config = Neo4jStoreConfig(
    auth_data=auth_data,
    batching=True,
    batch_size=5000,
    custom_prefixes=custom_prefixes  # 자동 추출된 결과물 투입
)

# 4. Neo4j 스토어 초기화 및 데이터 적재
neo4j_store = Neo4jStore(config=config)
g = Graph(store=neo4j_store)

print("GO Ontology 실제 임포트 시작... (시간이 소요됩니다)")
g.parse("data/go.owl", format="xml")
print("Neo4j로 임포트 완료!")

print("총 트리플(Triple) 개수:", len(g)) # len(g) = 0 맞음. 5천개씩 차면 서버에 저장하고 날려버리기 때문에

✅ 총 20개의 네임스페이스 매핑 완료!
Uniqueness constraint on :Resource(uri) found. 
                
                
The store is now: Open
GO Ontology 실제 임포트 시작... (시간이 소요됩니다)


Neo4j로 임포트 완료!
총 트리플(Triple) 개수: 0
